# 🖼️ ImageNet Classification & Error Calculation

This notebook provides a comprehensive explanation of the **ImageNet dataset**, **classification task**, and how to calculate **top-1** and **top-5 errors** with detailed examples.

---

## 📊 What is ImageNet Dataset?

**ImageNet** is one of the most important datasets in computer vision history, containing millions of labeled images across thousands of categories.

### **Key Statistics:**
- **Total Images**: ~14 million images
- **Categories**: 20,000+ categories (synsets)
- **ImageNet Classification (ILSVRC)**: Subset with 1,000 categories
- **Training Set**: ~1.2 million images
- **Validation Set**: 50,000 images (50 per class)
- **Test Set**: 100,000 images (100 per class)
- **Image Resolution**: Variable (typically resized to 224×224 for training)

### **Sample Categories:**
```
Class 0: tench (a type of fish)
Class 1: goldfish
Class 2: great white shark
Class 3: tiger shark
...
Class 281: tabby cat
Class 285: Egyptian cat
...
Class 999: toilet tissue
```

### **Importance:**
- **Benchmark Standard**: De facto standard for image classification
- **Model Comparison**: Enables fair comparison between architectures
- **Transfer Learning**: Pre-trained models serve as starting points
- **Research Driver**: Pushed development of CNN architectures

---

## 🎯 Classification Task & Error Metrics

### **Classification Task:**
Given an input image, predict which of the 1,000 classes it belongs to.

### **Model Output:**
- **Logits**: Raw scores for each class
- **Probabilities**: Softmax-normalized scores (sum to 1)
- **Predictions**: Classes ranked by probability

### **Error Metrics:**

#### **1. Top-1 Error Rate**
- **Definition**: Percentage of test samples where the **highest predicted class** is incorrect
- **Formula**: `Top-1 Error = (Number of incorrect top predictions / Total samples) × 100%`
- **Interpretation**: Strictest metric - only considers the most confident prediction

#### **2. Top-5 Error Rate**
- **Definition**: Percentage of test samples where the **correct class is NOT in the top 5 predictions**
- **Formula**: `Top-5 Error = (Number of samples with correct class outside top-5 / Total samples) × 100%`
- **Interpretation**: More lenient - allows model to be "partially correct"

### **Why Top-5 Error?**
- **Similar Categories**: Many ImageNet classes are visually similar (e.g., different dog breeds)
- **Ambiguous Images**: Some images could reasonably belong to multiple categories
- **Practical Relevance**: In real applications, showing top-5 results is often useful

---

In [ ]:
# Import required libraries
import numpy as np
import torch
import torch.nn.functional as F
from typing import List, Tuple
import matplotlib.pyplot as plt
import pandas as pd

print("📦 Libraries imported successfully!")

📦 Libraries imported successfully!


## 🧮 Mathematical Formulation

### **Notation:**
- `N`: Total number of test samples
- `y_i`: True class label for sample i
- `ŷ_i^(k)`: k-th highest predicted class for sample i
- `P_i`: Vector of predicted probabilities for sample i

### **Top-1 Error Rate:**
```
Top-1 Error = (1/N) × Σ(i=1 to N) 𝟙[y_i ≠ ŷ_i^(1)] × 100%
```
Where `𝟙[condition]` is the indicator function (1 if true, 0 if false)

### **Top-5 Error Rate:**
```
Top-5 Error = (1/N) × Σ(i=1 to N) 𝟙[y_i ∉ {ŷ_i^(1), ŷ_i^(2), ŷ_i^(3), ŷ_i^(4), ŷ_i^(5)}] × 100%
```

### **Top-k Accuracy (Alternative Formulation):**
```
Top-k Accuracy = (1/N) × Σ(i=1 to N) 𝟙[y_i ∈ {ŷ_i^(1), ..., ŷ_i^(k)}] × 100%
Top-k Error = 100% - Top-k Accuracy
```

---

In [ ]:
def calculate_top_k_accuracy(predictions: torch.Tensor,
                           targets: torch.Tensor,
                           k: int) -> float:
    """
    Calculate top-k accuracy for classification predictions.

    Args:
        predictions (torch.Tensor): Model predictions of shape (N, num_classes)
        targets (torch.Tensor): True labels of shape (N,)
        k (int): Number of top predictions to consider

    Returns:
        float: Top-k accuracy as percentage (0-100)
    """
    batch_size = targets.size(0)

    # Get top-k predictions
    _, top_k_preds = predictions.topk(k, dim=1, largest=True, sorted=True)

    # Check if true label is in top-k predictions
    targets_expanded = targets.view(-1, 1).expand_as(top_k_preds)
    correct = (top_k_preds == targets_expanded).any(dim=1)

    accuracy = correct.float().mean() * 100
    return accuracy.item()

def calculate_top_k_error(predictions: torch.Tensor,
                        targets: torch.Tensor,
                        k: int) -> float:
    """
    Calculate top-k error rate for classification predictions.

    Args:
        predictions (torch.Tensor): Model predictions of shape (N, num_classes)
        targets (torch.Tensor): True labels of shape (N,)
        k (int): Number of top predictions to consider

    Returns:
        float: Top-k error rate as percentage (0-100)
    """
    accuracy = calculate_top_k_accuracy(predictions, targets, k)
    return 100.0 - accuracy

print("✅ Error calculation functions defined!")

✅ Error calculation functions defined!


## 📝 Detailed Calculation Examples

Let's walk through concrete examples with step-by-step calculations.

### **Example 1: Single Sample**

**Scenario**: Model classifies an image of a "Golden Retriever" (class 207)

**Model Predictions (Top 5):**
1. **Labrador Retriever** (class 208): 0.35 probability
2. **Golden Retriever** (class 207): 0.28 probability ✅ (correct class)
3. **Beagle** (class 162): 0.15 probability
4. **Border Collie** (class 232): 0.12 probability
5. **German Shepherd** (class 235): 0.10 probability

**Calculations:**
- **Top-1 Error**: Predicted class 208 ≠ True class 207 → **ERROR** ❌
- **Top-5 Error**: True class 207 ∈ {208, 207, 162, 232, 235} → **CORRECT** ✅

**Results for this sample:**
- Top-1 Error: 1 (100%)
- Top-5 Error: 0 (0%)

---

In [ ]:
# Example 1: Single sample calculation
def example_single_sample():
    print("🔍 Example 1: Single Sample Calculation\n")

    # True label
    true_class = 207  # Golden Retriever

    # Model predictions (logits for 1000 classes)
    # We'll create a simplified example with just the top 5 predictions
    logits = torch.zeros(1000)  # Initialize all to 0
    logits[208] = 3.5  # Labrador Retriever (highest)
    logits[207] = 3.2  # Golden Retriever (second)
    logits[162] = 2.8  # Beagle
    logits[232] = 2.5  # Border Collie
    logits[235] = 2.2  # German Shepherd

    # Convert to probabilities
    probabilities = F.softmax(logits, dim=0)

    # Get top 5 predictions
    top5_probs, top5_classes = torch.topk(probabilities, 5)

    print("📊 Model Predictions (Top 5):")
    class_names = ['Labrador Retriever', 'Golden Retriever', 'Beagle', 'Border Collie', 'German Shepherd']
    for i, (cls, prob) in enumerate(zip(top5_classes, top5_probs)):
        marker = "✅" if cls.item() == true_class else "❌"
        print(f"{i+1}. Class {cls.item()}: {class_names[i]} - {prob.item():.3f} {marker}")

    # Calculate errors
    predictions = logits.unsqueeze(0)  # Add batch dimension
    targets = torch.tensor([true_class])

    top1_error = calculate_top_k_error(predictions, targets, k=1)
    top5_error = calculate_top_k_error(predictions, targets, k=5)

    print(f"\n📈 Results:")
    print(f"Top-1 Error: {top1_error:.1f}% (predicted class {top5_classes[0].item()} ≠ true class {true_class})")
    print(f"Top-5 Error: {top5_error:.1f}% (true class {true_class} ∈ top-5 predictions)")

    return top1_error, top5_error

example_single_sample()

🔍 Example 1: Single Sample Calculation

📊 Model Predictions (Top 5):
1. Class 208: Labrador Retriever - 0.030 ❌
2. Class 207: Golden Retriever - 0.023 ✅
3. Class 162: Beagle - 0.015 ❌
4. Class 232: Border Collie - 0.011 ❌
5. Class 235: German Shepherd - 0.008 ❌

📈 Results:
Top-1 Error: 100.0% (predicted class 208 ≠ true class 207)
Top-5 Error: 0.0% (true class 207 ∈ top-5 predictions)


(100.0, 0.0)

### **Example 2: Batch of Samples**

Let's calculate errors for a batch of 5 samples with different scenarios.

In [ ]:
def example_batch_calculation():
    print("🔍 Example 2: Batch Calculation (5 Samples)\n")

    # Setup batch of 5 samples
    batch_size = 5
    num_classes = 1000

    # True labels
    true_labels = torch.tensor([285, 207, 1, 999, 500])  # Egyptian cat, Golden Retriever, Goldfish, Toilet tissue, Fire truck
    class_names = ['Egyptian cat', 'Golden Retriever', 'Goldfish', 'Toilet tissue', 'Fire truck']

    # Create sample predictions for each image
    predictions = torch.randn(batch_size, num_classes)  # Random logits

    # Manually set some predictions to create different scenarios

    # Sample 1: Top-1 correct, Top-5 correct
    predictions[0, 285] = 5.0  # Egyptian cat (correct) - highest
    predictions[0, 281] = 4.5  # Tabby cat - second

    # Sample 2: Top-1 wrong, Top-5 correct (Golden Retriever example from before)
    predictions[1, 208] = 4.8  # Labrador Retriever - highest (wrong)
    predictions[1, 207] = 4.5  # Golden Retriever - second (correct)

    # Sample 3: Top-1 wrong, Top-5 wrong
    predictions[2, 150] = 4.5  # Some other class - highest (wrong)
    predictions[2, 200] = 4.2  # Another wrong class
    predictions[2, 1] = 3.0    # Goldfish - 6th highest (correct but outside top-5)

    # Sample 4: Top-1 correct, Top-5 correct
    predictions[3, 999] = 5.2  # Toilet tissue (correct) - highest

    # Sample 5: Top-1 wrong, Top-5 correct
    predictions[4, 300] = 4.8  # Wrong class - highest
    predictions[4, 500] = 4.3  # Fire truck - third (correct)

    # Calculate errors
    top1_error = calculate_top_k_error(predictions, true_labels, k=1)
    top5_error = calculate_top_k_error(predictions, true_labels, k=5)

    print("📊 Sample-by-Sample Analysis:")
    print("-" * 60)

    for i in range(batch_size):
        # Get top 5 for this sample
        top5_probs, top5_classes = torch.topk(predictions[i], 5)
        true_class = true_labels[i].item()

        # Check if correct
        top1_correct = top5_classes[0].item() == true_class
        top5_correct = true_class in top5_classes.tolist()

        print(f"Sample {i+1}: {class_names[i]} (class {true_class})")
        print(f"  Top-1 prediction: class {top5_classes[0].item()} {'✅' if top1_correct else '❌'}")
        print(f"  Top-5 predictions: {top5_classes.tolist()} {'✅' if top5_correct else '❌'}")
        print(f"  Top-1 error: {'0%' if top1_correct else '100%'}")
        print(f"  Top-5 error: {'0%' if top5_correct else '100%'}")
        print()

    print(f"📈 Overall Batch Results:")
    print(f"Top-1 Error Rate: {top1_error:.1f}%")
    print(f"Top-5 Error Rate: {top5_error:.1f}%")

    # Manual calculation verification
    print(f"\n🔍 Manual Verification:")
    top1_errors = 0
    top5_errors = 0

    for i in range(batch_size):
        top5_probs, top5_classes = torch.topk(predictions[i], 5)
        true_class = true_labels[i].item()

        if top5_classes[0].item() != true_class:
            top1_errors += 1
        if true_class not in top5_classes.tolist():
            top5_errors += 1

    manual_top1_error = (top1_errors / batch_size) * 100
    manual_top5_error = (top5_errors / batch_size) * 100

    print(f"Manual Top-1 Error: {top1_errors}/{batch_size} = {manual_top1_error:.1f}%")
    print(f"Manual Top-5 Error: {top5_errors}/{batch_size} = {manual_top5_error:.1f}%")

    return top1_error, top5_error

example_batch_calculation()

🔍 Example 2: Batch Calculation (5 Samples)

📊 Sample-by-Sample Analysis:
------------------------------------------------------------
Sample 1: Egyptian cat (class 285)
  Top-1 prediction: class 285 ✅
  Top-5 predictions: [285, 281, 986, 270, 434] ✅
  Top-1 error: 0%
  Top-5 error: 0%

Sample 2: Golden Retriever (class 207)
  Top-1 prediction: class 208 ❌
  Top-5 predictions: [208, 207, 747, 264, 662] ✅
  Top-1 error: 100%
  Top-5 error: 0%

Sample 3: Goldfish (class 1)
  Top-1 prediction: class 150 ❌
  Top-5 predictions: [150, 200, 737, 1, 631] ✅
  Top-1 error: 100%
  Top-5 error: 0%

Sample 4: Toilet tissue (class 999)
  Top-1 prediction: class 999 ✅
  Top-5 predictions: [999, 315, 514, 621, 764] ✅
  Top-1 error: 0%
  Top-5 error: 0%

Sample 5: Fire truck (class 500)
  Top-1 prediction: class 300 ❌
  Top-5 predictions: [300, 500, 713, 552, 28] ✅
  Top-1 error: 100%
  Top-5 error: 0%

📈 Overall Batch Results:
Top-1 Error Rate: 60.0%
Top-5 Error Rate: 0.0%

🔍 Manual Verification:
Manua

(60.0, 0.0)

## 🛠️ Practical Implementation

Here's how you would implement error calculation in a real training/evaluation loop.

In [ ]:
class ImageNetErrorCalculator:
    """
    A comprehensive error calculator for ImageNet classification.

    This class provides methods to calculate and track top-1 and top-5 errors
    during model evaluation on ImageNet or similar classification datasets.

    Attributes:
        reset(): Reset all accumulated statistics
        update(predictions, targets): Update statistics with new batch
        compute(): Compute final error rates
    """

    def __init__(self):
        self.reset()

    def reset(self):
        """Reset all accumulated statistics."""
        self.total_samples = 0
        self.top1_correct = 0
        self.top5_correct = 0

    def update(self, predictions: torch.Tensor, targets: torch.Tensor):
        """
        Update error statistics with a new batch of predictions.

        Args:
            predictions (torch.Tensor): Model predictions of shape (batch_size, num_classes)
            targets (torch.Tensor): True labels of shape (batch_size,)
        """
        batch_size = targets.size(0)
        self.total_samples += batch_size

        # Calculate top-1 accuracy
        _, top1_pred = predictions.max(dim=1)
        self.top1_correct += (top1_pred == targets).sum().item()

        # Calculate top-5 accuracy
        _, top5_pred = predictions.topk(5, dim=1)
        targets_expanded = targets.view(-1, 1).expand_as(top5_pred)
        self.top5_correct += (top5_pred == targets_expanded).any(dim=1).sum().item()

    def compute(self) -> Tuple[float, float, float, float]:
        """
        Compute final accuracy and error rates.

        Returns:
            Tuple[float, float, float, float]: (top1_accuracy, top5_accuracy, top1_error, top5_error)
        """
        if self.total_samples == 0:
            return 0.0, 0.0, 100.0, 100.0

        top1_accuracy = (self.top1_correct / self.total_samples) * 100
        top5_accuracy = (self.top5_correct / self.total_samples) * 100
        top1_error = 100.0 - top1_accuracy
        top5_error = 100.0 - top5_accuracy

        return top1_accuracy, top5_accuracy, top1_error, top5_error

    def __str__(self) -> str:
        """String representation of current statistics."""
        top1_acc, top5_acc, top1_err, top5_err = self.compute()
        return f"Samples: {self.total_samples} | Top-1: {top1_acc:.2f}% | Top-5: {top5_acc:.2f}% | Top-1 Error: {top1_err:.2f}% | Top-5 Error: {top5_err:.2f}%"

# Example usage
def demo_error_calculator():
    print("🛠️ ImageNet Error Calculator Demo\n")

    calculator = ImageNetErrorCalculator()

    # Simulate evaluation over multiple batches
    num_batches = 3
    batch_size = 4
    num_classes = 1000

    print("Processing batches:")
    for batch_idx in range(num_batches):
        # Generate random predictions and targets
        predictions = torch.randn(batch_size, num_classes)
        targets = torch.randint(0, num_classes, (batch_size,))

        # Manually set some correct predictions for demonstration
        if batch_idx == 0:
            # Make first sample top-1 correct
            predictions[0, targets[0]] = 10.0
            # Make second sample top-5 correct but not top-1
            predictions[1, targets[1]] = 5.0
            predictions[1, targets[1] + 1] = 6.0

        calculator.update(predictions, targets)

        top1_acc, top5_acc, top1_err, top5_err = calculator.compute()
        print(f"  Batch {batch_idx + 1}: {calculator}")

    print(f"\n📈 Final Results:")
    top1_acc, top5_acc, top1_err, top5_err = calculator.compute()
    print(f"Top-1 Accuracy: {top1_acc:.2f}%")
    print(f"Top-5 Accuracy: {top5_acc:.2f}%")
    print(f"Top-1 Error Rate: {top1_err:.2f}%")
    print(f"Top-5 Error Rate: {top5_err:.2f}%")

demo_error_calculator()

🛠️ ImageNet Error Calculator Demo

Processing batches:
  Batch 1: Samples: 4 | Top-1: 25.00% | Top-5: 50.00% | Top-1 Error: 75.00% | Top-5 Error: 50.00%
  Batch 2: Samples: 8 | Top-1: 12.50% | Top-5: 25.00% | Top-1 Error: 87.50% | Top-5 Error: 75.00%
  Batch 3: Samples: 12 | Top-1: 8.33% | Top-5: 16.67% | Top-1 Error: 91.67% | Top-5 Error: 83.33%

📈 Final Results:
Top-1 Accuracy: 8.33%
Top-5 Accuracy: 16.67%
Top-1 Error Rate: 91.67%
Top-5 Error Rate: 83.33%


## 📚 Key Takeaways

### **ImageNet Dataset:**
- **Scale**: 1.2M training images, 1000 classes
- **Importance**: Standard benchmark for computer vision
- **Impact**: Drove development of modern CNN architectures

### **Error Metrics:**
- **Top-1 Error**: Strictest metric - only top prediction matters
- **Top-5 Error**: More lenient - allows for reasonable alternatives
- **Why Both**: Different use cases and practical considerations

### **Calculation Process:**
1. Get model predictions (logits or probabilities)
2. Sort predictions by confidence (highest first)
3. Check if true label is in top-k predictions
4. Calculate percentage of incorrect predictions

### **Historical Progress:**
- **2012**: AlexNet sparked deep learning revolution (15.3% top-5 error)
- **2015**: ResNet achieved human-level performance (~5% top-5 error)
- **2021**: Vision Transformers pushed further (2.1% top-5 error)

### **Practical Implementation:**
- Use batch processing for efficiency
- Track statistics across entire validation set
- Report both accuracy and error rates
- Consider top-5 for user-facing applications

---

## 🔗 References

1. **ImageNet Paper**: Deng et al., "ImageNet: A Large-Scale Hierarchical Image Database" (CVPR 2009)
2. **AlexNet Paper**: Krizhevsky et al., "ImageNet Classification with Deep CNNs" (NIPS 2012)
3. **ImageNet Website**: http://www.image-net.org/
4. **ILSVRC Competition**: http://www.image-net.org/challenges/LSVRC/

---

*This notebook provides a comprehensive understanding of ImageNet classification and error calculation methods used in computer vision research.*